<a href="https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)
user_info = api.whoami()

print("✅ Hugging Face token is working")
print("Logged in as:", user_info["name"])

✅ Hugging Face token is working
Logged in as: Krishna127


In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("✅ FlyRank dataset access is working")
print("Number of files:", len(files))

print("\nFirst 20 files:")
for file in files[:20]:
    print(file)

✅ FlyRank dataset access is working
Number of files: 24

First 20 files:
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_per

In [ ]:
!pip -q install huggingface_hub pyarrow

import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata

# Read the token safely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

REPO_ID = "FlyRank/internship-warehouse"

# Connect to the dataset
api = HfApi(token=HF_TOKEN)

# Get the list of files
files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

print("Total files found:", len(files))

# Download and load dim_clients
clients_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=HF_TOKEN
)

clients_df = pd.read_parquet(clients_path)

# Download and load dim_content
content_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="dim_content.parquet",
    token=HF_TOKEN
)

content_df = pd.read_parquet(content_path)

# Find daily performance files
daily_files = sorted([
    file for file in files
    if "fact_content_daily_performance" in file
    and file.endswith(".parquet")
])

print("Daily files found:", len(daily_files))
print("First daily file:", daily_files[0])

# Download and load one daily file for verification
daily_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename=daily_files[0],
    token=HF_TOKEN
)

daily_df = pd.read_parquet(daily_path)

# Check that all DataFrames are loaded
print("\n✅ Data loaded successfully")
print("clients_df:", clients_df.shape)
print("content_df:", content_df.shape)
print("daily_df:", daily_df.shape)

print("\nDaily table columns:")
print(daily_df.columns.tolist())

Total files found: 24
Daily files found: 19
First daily file: fact_content_daily_performance/month=2025-01/data_0.parquet

✅ Data loaded successfully
clients_df: (104, 9)
content_df: (519606, 26)
daily_df: (1297, 30)

Daily table columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

**Unit of analysis:** One row represents the measured daily performance of one pseudonymized content item for one pseudonymized client on one report date.

**Time window:** The currently loaded partition contains observations for January 2025. The complete warehouse contains multiple monthly partitions, so the full observed date range will be verified after all daily partitions are loaded.

The daily performance table provides content-level measurements from Google Search Console (GSC) and Google Analytics 4 (GA4). Data availability flags are used to distinguish unavailable data from measured zero values.

In [ ]:
# Section 1: Verify the grain and time window

# Convert the date column to datetime
daily_df["report_date"] = pd.to_datetime(daily_df["report_date"])

# Verify the observed date range
print("Observed time window:")
print("Start date:", daily_df["report_date"].min())
print("End date:", daily_df["report_date"].max())

# Verify the proposed grain:
# one row per client, content item, and report date
grain_columns = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

duplicate_rows = daily_df.duplicated(
    subset=grain_columns
).sum()

print("\nTotal rows:", len(daily_df))
print("Duplicate client-content-date combinations:", duplicate_rows)

# Count unique clients and content items
print("\nUnique clients:", daily_df["client_hash_id"].nunique())
print("Unique content items:", daily_df["content_hash_id"].nunique())

# Display a few rows used to verify the grain
display(
    daily_df[
        [
            "report_date",
            "client_hash_id",
            "content_hash_id"
        ]
    ].head()
)

Observed time window:
Start date: 2025-01-27 00:00:00
End date: 2025-01-31 00:00:00

Total rows: 1297
Duplicate client-content-date combinations: 0

Unique clients: 2
Unique content items: 476


,report_date,client_hash_id,content_hash_id
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features

The following fields will be used as input features for SEO content prioritization:

- `gsc_impressions` — measures how often a content item appeared in Google Search results.
- `gsc_clicks` — measures organic search clicks received by the content item.
- `gsc_avg_position` — measures the observed average search position.
- `ga4_pageviews` — measures page views recorded in GA4.
- `ga4_sessions` — measures sessions associated with the content item.
- `ga4_users` — measures users associated with the content item.
- `ga4_engaged_sessions` — measures engaged sessions.
- `ga4_total_engagement_sec` — measures total recorded engagement time.
- `sessions_organic` — measures sessions attributed to organic traffic.
- `sessions_direct` — measures direct sessions.
- `sessions_referral` — measures referral sessions.
- `sessions_social` — measures social sessions.
- Content metadata from `dim_content`, such as content type, search intent, content age, and freshness-related fields, will be used when available.

These fields describe observed search visibility, traffic, engagement, traffic sources, and content characteristics. They can help identify content items that may deserve human review or updating.

### Label / decision target

There is no directly observed `refresh` label in the warehouse. Therefore, the decision target is a **refresh-priority score or ranking** created from observed performance signals.

Content may receive a higher priority when it shows a combination of declining or weak performance, meaningful search visibility, engagement opportunity, and signs of content age or staleness.

This is a decision-support target. It does not prove that refreshing a content item will improve its future performance.

### Context fields

- `report_date` — identifies the date of the daily observation and supports time-window analysis.
- `client_hash_id` — identifies the pseudonymized client and supports client-level grouping.
- `content_hash_id` — identifies the pseudonymized content item and supports joins with the content table.
- `client_has_gsc` — indicates whether the client has GSC data coverage.
- `client_has_ga4` — indicates whether the client has GA4 data coverage.
- `gsc_data_available` — indicates whether GSC data is available for the observation.
- `ga4_data_available` — indicates whether GA4 data is available for the observation.

These fields provide identification, grouping, time, and data-availability context. They are not treated as direct performance signals.

### Excluded fields

- Real client names are excluded because the analysis does not require client identity and the warehouse uses pseudonymized identifiers.
- URLs are excluded because they are not required for the prioritization decision and should not be exposed in the notebook.
- Private search queries are excluded to protect privacy and because this contract focuses on content-level performance.
- `gsc_sum_position` is excluded because it is an intermediate aggregate used to calculate average position. Using both `gsc_sum_position` and `gsc_avg_position` may duplicate the same ranking signal.
- Future observations are excluded from features to prevent target leakage.
- Fields with unavailable GSC or GA4 history are not interpreted as measured zero values. The corresponding availability flags must be checked before using those metrics.

In [ ]:
# Section 2: Verify that the planned fields exist

feature_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social"
]

context_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available"
]

excluded_fields = [
    "gsc_sum_position"
]

all_planned_fields = (
    feature_fields
    + context_fields
    + excluded_fields
)

print("Field availability check:\n")

for field in all_planned_fields:
    if field in daily_df.columns:
        print(f"✅ {field}")
    else:
        print(f"❌ {field} — not found in daily_df")

print("\nNumber of planned fields:", len(all_planned_fields))
print(
    "Number found:",
    sum(
        field in daily_df.columns
        for field in all_planned_fields
    )
)

Field availability check:

✅ gsc_impressions
✅ gsc_clicks
✅ gsc_avg_position
✅ ga4_pageviews
✅ ga4_sessions
✅ ga4_users
✅ ga4_engaged_sessions
✅ ga4_total_engagement_sec
✅ sessions_organic
✅ sessions_direct
✅ sessions_referral
✅ sessions_social
✅ report_date
✅ client_hash_id
✅ content_hash_id
✅ client_has_gsc
✅ client_has_ga4
✅ gsc_data_available
✅ ga4_data_available
✅ gsc_sum_position

Number of planned fields: 20
Number found: 20


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 3. Verify it with queries

The following queries verify the main claims in this data contract:

1. The daily table is checked for duplicate combinations of `client_hash_id`, `content_hash_id`, and `report_date` to verify the proposed grain.
2. The minimum and maximum `report_date` values are measured to verify the observed time window.
3. Row counts and unique client/content counts are measured to describe the loaded data.
4. Missing-value counts are calculated for all planned feature and context fields.
5. GSC and GA4 availability flags are checked so unavailable data is not incorrectly interpreted as a measured value of zero.
6. Observation counts are summarized by client and by date to identify uneven data coverage.

These checks provide evidence for the contract claims. They do not establish that the data is complete or causally explain performance changes.

In [ ]:
# Section 3: Verify grain, counts, missing values, and data availability

# Make sure the report date is in datetime format
daily_df["report_date"] = pd.to_datetime(
    daily_df["report_date"]
)

# -------------------------------------------------
# 1. Verify the proposed grain
# -------------------------------------------------

grain_columns = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

duplicate_rows = daily_df.duplicated(
    subset=grain_columns
).sum()

print("1. GRAIN CHECK")
print("Total rows:", len(daily_df))
print(
    "Duplicate client-content-date combinations:",
    duplicate_rows
)

# -------------------------------------------------
# 2. Verify the observed time window
# -------------------------------------------------

print("\n2. TIME-WINDOW CHECK")
print(
    "First observed date:",
    daily_df["report_date"].min()
)
print(
    "Last observed date:",
    daily_df["report_date"].max()
)

# -------------------------------------------------
# 3. Verify counts
# -------------------------------------------------

print("\n3. COUNT CHECK")
print(
    "Unique clients:",
    daily_df["client_hash_id"].nunique()
)
print(
    "Unique content items:",
    daily_df["content_hash_id"].nunique()
)
print(
    "Unique report dates:",
    daily_df["report_date"].nunique()
)

# -------------------------------------------------
# 4. Check missing values
# -------------------------------------------------

planned_fields = (
    feature_fields
    + context_fields
)

print("\n4. MISSING-VALUE CHECK")

missing_summary = pd.DataFrame({
    "missing_count":
        daily_df[planned_fields].isna().sum(),

    "missing_percent":
        (
            daily_df[planned_fields]
            .isna()
            .mean()
            * 100
        ).round(2)
})

display(
    missing_summary.sort_values(
        "missing_count",
        ascending=False
    )
)

# -------------------------------------------------
# 5. Check GSC and GA4 availability
# -------------------------------------------------

print("\n5. DATA-AVAILABILITY CHECK")

print("\nGSC availability:")
print(
    daily_df["gsc_data_available"]
    .value_counts(
        dropna=False
    )
)

print("\nGA4 availability:")
print(
    daily_df["ga4_data_available"]
    .value_counts(
        dropna=False
    )
)

# -------------------------------------------------
# 6. Check coverage by client
# -------------------------------------------------

print("\n6. CLIENT-COVERAGE CHECK")

client_coverage = (
    daily_df
    .groupby("client_hash_id")
    .agg(
        rows=(
            "report_date",
            "size"
        ),
        first_date=(
            "report_date",
            "min"
        ),
        last_date=(
            "report_date",
            "max"
        ),
        unique_content_items=(
            "content_hash_id",
            "nunique"
        )
    )
)

display(
    client_coverage
    .sort_values(
        "rows",
        ascending=False
    )
)

# -------------------------------------------------
# 7. Check coverage by date
# -------------------------------------------------

print("\n7. DAILY-COVERAGE CHECK")

daily_coverage = (
    daily_df
    .groupby("report_date")
    .agg(
        rows=(
            "content_hash_id",
            "size"
        ),
        clients=(
            "client_hash_id",
            "nunique"
        ),
        content_items=(
            "content_hash_id",
            "nunique"
        )
    )
)

display(daily_coverage)

1. GRAIN CHECK
Total rows: 1297
Duplicate client-content-date combinations: 0

2. TIME-WINDOW CHECK
First observed date: 2025-01-27 00:00:00
Last observed date: 2025-01-31 00:00:00

3. COUNT CHECK
Unique clients: 2
Unique content items: 476
Unique report dates: 5

4. MISSING-VALUE CHECK


,missing_count,missing_percent
gsc_impressions,0,0.0
gsc_clicks,0,0.0
gsc_avg_position,0,0.0
ga4_pageviews,0,0.0
ga4_sessions,0,0.0
ga4_users,0,0.0
ga4_engaged_sessions,0,0.0
ga4_total_engagement_sec,0,0.0
sessions_organic,0,0.0
sessions_direct,0,0.0



5. DATA-AVAILABILITY CHECK

GSC availability:
gsc_data_available
True    1297
Name: count, dtype: int64

GA4 availability:
ga4_data_available
False    1297
Name: count, dtype: int64

6. CLIENT-COVERAGE CHECK


,rows,first_date,last_date,unique_content_items
client_hash_id,,,,
client_9958f0a7ae1df715,770,2025-01-27,2025-01-31,173
client_ff644d8251367cbb,527,2025-01-27,2025-01-31,303



7. DAILY-COVERAGE CHECK


,rows,clients,content_items
report_date,,,
2025-01-27,303,2,303
2025-01-28,317,2,317
2025-01-29,262,2,262
2025-01-30,194,2,194
2025-01-31,221,2,221


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits

This dataset supports **observed, measured, and directional decision-support analysis**, but it cannot prove causation.

### Unbalanced history

Client and content histories may have different start dates, end dates, and levels of coverage. Therefore, raw performance totals are not automatically comparable across all clients or content items. A content item with fewer observations may appear to perform worse simply because less history is available.

### GSC-only early rows

Some early observations may have GSC data while GA4 data is unavailable. In these cases, search visibility metrics such as impressions, clicks, and average position may be observed, but pageviews, sessions, users, and engagement metrics may be missing or unavailable.

Therefore, unavailable GA4 data must not be interpreted as zero traffic or zero engagement. The `gsc_data_available` and `ga4_data_available` fields must be checked before comparing metrics.

### Window overlaps

Some performance measures may be calculated using rolling or overlapping time windows. Overlapping windows share observations, so they are not independent measurements. Changes between overlapping windows should be interpreted as directional evidence rather than separate, independent proof of a performance change.

### What the data cannot tell us

This dataset cannot tell us:

- Whether refreshing a specific content item will definitely improve future traffic, rankings, or engagement.
- Whether a performance decline was caused by outdated content rather than seasonality, search-demand changes, competition, technical issues, or other external factors.
- The editorial effort, cost, or business value associated with refreshing a content item.
- The full quality, accuracy, usefulness, or user experience of the content.
- Whether a high-priority recommendation will produce a positive outcome after a refresh.

The data can help **prioritize content for human review**, but it should not automatically decide which content must be updated.

In [ ]:
# Section 4: Check data-history and availability limits

# Convert report_date to datetime
daily_df["report_date"] = pd.to_datetime(
    daily_df["report_date"]
)

# -------------------------------------------------
# 1. Check history coverage by client
# -------------------------------------------------

client_history = (
    daily_df
    .groupby("client_hash_id")
    .agg(
        first_observed_date=(
            "report_date",
            "min"
        ),
        last_observed_date=(
            "report_date",
            "max"
        ),
        observation_days=(
            "report_date",
            "nunique"
        ),
        observation_rows=(
            "report_date",
            "size"
        )
    )
)

print("CLIENT HISTORY COVERAGE")
display(
    client_history
    .sort_values(
        "observation_days"
    )
)

print("\nHistory coverage summary:")
display(
    client_history[
        [
            "observation_days",
            "observation_rows"
        ]
    ].describe()
)

# -------------------------------------------------
# 2. Check GSC and GA4 availability combinations
# -------------------------------------------------

availability_check = (
    daily_df
    .groupby(
        [
            "gsc_data_available",
            "ga4_data_available"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="row_count"
    )
)

print("\nGSC AND GA4 AVAILABILITY COMBINATIONS")
display(availability_check)

# -------------------------------------------------
# 3. Compare missing values with availability
# -------------------------------------------------

print("\nMISSING VALUES BY DATA AVAILABILITY")

availability_missing = (
    daily_df
    .groupby(
        [
            "gsc_data_available",
            "ga4_data_available"
        ],
        dropna=False
    )
    [
        [
            "gsc_impressions",
            "gsc_clicks",
            "ga4_sessions",
            "ga4_pageviews",
            "ga4_engaged_sessions"
        ]
    ]
    .apply(
        lambda x: x.isna().mean() * 100
    )
    .round(2)
)

display(availability_missing)

CLIENT HISTORY COVERAGE


,first_observed_date,last_observed_date,observation_days,observation_rows
client_hash_id,,,,
client_9958f0a7ae1df715,2025-01-27,2025-01-31,5,770
client_ff644d8251367cbb,2025-01-27,2025-01-31,5,527



History coverage summary:


,observation_days,observation_rows
count,2.0,2.000000
mean,5.0,648.500000
std,0.0,171.826948
min,5.0,527.000000
25%,5.0,587.750000
50%,5.0,648.500000
75%,5.0,709.250000
max,5.0,770.000000



GSC AND GA4 AVAILABILITY COMBINATIONS


,gsc_data_available,ga4_data_available,row_count
0,True,False,1297



MISSING VALUES BY DATA AVAILABILITY


,,gsc_impressions,gsc_clicks,ga4_sessions,ga4_pageviews,ga4_engaged_sessions
gsc_data_available,ga4_data_available,,,,,
True,False,0.0,0.0,0.0,0.0,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.